# 02. DeOldify 파이프라인

**Paper / Repo**: https://github.com/jantic/DeOldify (jantic, Artistic / Stable / Video 3종)

## 결과 요약 (보고서 기준)
* Pre-train 된 Artistic 모델로 한국 음식 inference 까지는 성공.
* fine-tune (`ColorizeTrainingArtistic`) 시도 → fastai/torch 버전 호환 + 메모리 이슈로 진행 실패.
* 본 노트북은 (1) 원본 repo clone (2) 사전학습 weight 로 한국 음식 inference (3) fine-tune 시도 코드를 정리합니다.

## 1. 환경 셋업

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/딥러닝
!git clone https://github.com/jantic/DeOldify.git DeOldify_run || true
%cd DeOldify_run
!pip install -q -r requirements-colab.txt || pip install -q fastai==1.0.60 yt-dlp ffmpeg-python tensorboardX

## 2. 사전학습 weight 다운로드

Artistic 가중치 (~250MB).

In [ ]:
import os
os.makedirs('models', exist_ok=True)
!wget -q -O models/ColorizeArtistic_gen.pth https://data.deepai.org/deoldify/ColorizeArtistic_gen.pth || \
  wget -q -O models/ColorizeArtistic_gen.pth https://huggingface.co/databuzzword/deoldify-artistic/resolve/main/ColorizeArtistic_gen.pth

## 3. 한국 음식 흑백 이미지 colorize

In [ ]:
import torch
from deoldify import device
from deoldify.device_id import DeviceId
device.set(device=DeviceId.GPU0 if torch.cuda.is_available() else DeviceId.CPU)
from deoldify.visualize import get_image_colorizer
colorizer = get_image_colorizer(artistic=True)

In [ ]:
import glob, os, shutil
from PIL import Image
from pathlib import Path
out_dir = Path('Output_korean_food'); out_dir.mkdir(exist_ok=True)
test_gray = sorted(glob.glob('../DATASET/test/gray_image/*.png'))[:30]
for gp in test_gray:
    img = colorizer.get_transformed_image(gp, render_factor=24)
    img.save(out_dir / os.path.basename(gp).replace('_gray.png', '_pred.png'))
print('saved:', len(list(out_dir.glob('*.png'))))

## 4. Fine-tune 시도 (참고)

원본 `ColorizeTrainingArtistic.ipynb` 의 `colorize_crit_learner` 학습 흐름을 한국 음식 데이터셋에 맞게 경로만 바꿔서 호출합니다.
보고서 시점에는 fastai 의존성 / 메모리 부족으로 학습이 끝까지 진행되지 못했습니다.

In [ ]:
# from fastai.vision import *
# from deoldify.generators import gen_inference_deep
# from deoldify.critics import colorize_crit_learner
# data_path = Path('../DATASET/train')
# learn_gen = gen_inference_deep(root_folder=data_path, weights_name='ColorizeArtistic_gen')
# learn_gen.unfreeze()
# learn_gen.fit_one_cycle(1, max_lr=1e-5)  # OOM 발생 지점
# learn_gen.save('ColorizeArtistic_gen_korean')

## 메모

* 본 노트북은 보고서의 'pre-trained inference 가능, fine-tune 실패' 결과를 코드로 재현하기 위한 정리입니다.
* fine-tune 셀은 의도적으로 주석 처리 — 실제 실행 시 fastai 1.x 환경 + 더 큰 GPU 가 필요합니다.